# Notebook 02 — Refined Dataset Effect (mBERT, 60.6%)

## Experiment Overview

This notebook quantifies the effect of **data quality improvement alone** on model performance.
We train the exact same mBERT architecture as Notebook 01, but replace the original noisy
labels with our human-verified Refined Dataset.

The Refined Dataset was built by:
1. Removing 376 duplicate sample IDs (752 entries total) — annotation ambiguity
2. Manually reviewing and correcting **763 labels** out of 5,567 remaining samples (~13.7%)

This corresponds to **Experiment 11** in `docs/experiment_log.md`.

## Why These Settings?

All hyperparameters are kept **identical** to Notebook 01 to isolate the data quality effect.

| Hyperparameter | Value | Note |
|---|---|---|
| Model | `bert-base-multilingual-cased` | Same as baseline |
| Learning rate | `1e-5` | Same as baseline |
| Epochs | `5` | Same as baseline |
| Batch size | `16` | Same as baseline |
| Class weights | `[3.25, 1.96, 1.0]` | Same as baseline |

## Result

**Validation accuracy: 60.6% (+4.0 pp over baseline)**

A +4.0 percentage point gain from data cleaning alone — larger than any architectural
improvement attempted in Experiments 1–10. This is the key finding of the paper:
**data quality > model complexity**.

> **Before running this notebook**, generate the Refined Dataset:
> ```
> cd data && python build_refined_dataset.py
> ```

In [ ]:
# Install dependencies (run once)
!pip install transformers datasets accelerate -q

import json
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")

# Same hyperparameters as Notebook 01 — intentionally unchanged
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128
BATCH_SIZE = 16
LR         = 1e-5
NUM_EPOCHS = 5
TEST_SIZE  = 0.2
SEED       = 42
OUTPUT_DIR = "./results/mbert_refined"

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {0: "negative", 1: "neutral", 2: "positive"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading

We load the Refined Dataset generated by `data/build_refined_dataset.py`.
Run that script once before this notebook.

The Refined Dataset (5,567 samples) is smaller than the original (6,319) because
duplicate sample IDs — which indicate the same tweet annotated multiple times with
inconsistent labels — were removed entirely.

In [ ]:
# Path to the Refined Dataset produced by data/build_refined_dataset.py
DATA_PATH = "../data/refined_dataset.json"

print(f"Loading Refined Dataset from {DATA_PATH} ...")
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

metadata = raw.get("metadata", {})
print(f"Dataset: {metadata.get('dataset', 'unknown')}")
print(f"Total samples: {metadata.get('total_samples', '?'):,}")
print(f"Label corrections applied: {metadata.get('label_corrections_applied', '?')}")

records = []
for item in raw["data"]:
    label = item["sentiment"]
    if label not in LABEL2ID:
        continue
    records.append({"text": item["text"], "label": label})

df = pd.DataFrame(records)
df["label_id"] = df["label"].map(LABEL2ID)

print(f"\nUsable samples: {len(df):,}")
print("Label distribution:")
for lbl, cnt in df["label"].value_counts().items():
    print(f"  {lbl}: {cnt:,} ({cnt / len(df) * 100:.1f}%)")

# Stratified 80/20 split — same ratio and seed as Notebook 01
train_df, val_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label"]
)
print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SentimentDataset(train_df["text"].tolist(), train_df["label_id"].tolist(), tokenizer, MAX_LENGTH)
val_dataset   = SentimentDataset(val_df["text"].tolist(),   val_df["label_id"].tolist(),   tokenizer, MAX_LENGTH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)
print("Data and model ready.")

## 2. Training

Training configuration is **identical to Notebook 01**. Any difference in the final result
is solely attributable to the improved label quality.

In [ ]:
class_weights = torch.tensor([3.25, 1.96, 1.0]).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        loss    = nn.CrossEntropyLoss(weight=class_weights)(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Training ...")
trainer.train()

results   = trainer.evaluate()
final_acc = results["eval_accuracy"]
train_logs = trainer.state.log_history
print(f"\nFinal validation accuracy: {final_acc:.4f} ({final_acc * 100:.2f}%)")
print(f"Improvement over baseline:  +{(final_acc - 0.566) * 100:.2f} pp")

## 3. Evaluation

**Result: 60.6% accuracy (+4.0 pp over baseline)**

Key observations:
- The gain comes entirely from cleaner labels — no architectural change was made.
- Training curves are noticeably smoother compared to Notebook 01, suggesting that
  the model receives more consistent gradient signals.
- Accuracy peaks around epoch 4.5 (same as the baseline), confirming that the
  optimal training duration is a property of the model–data pair, not just the data.
- Experiments 12–14 varied epochs, learning rate, and class weights around this result
  — none produced a meaningful improvement beyond 60.6%.

In [ ]:
# Detailed classification report
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=list(ID2LABEL.values())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(ID2LABEL.values()),
            yticklabels=list(ID2LABEL.values()))
plt.title(f"mBERT + Refined Dataset — Accuracy: {final_acc:.3f}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Loss and accuracy curves
train_steps, train_losses = [], []
eval_steps,  eval_losses, eval_accs = [], [], []
for log in train_logs:
    if "loss" in log and "eval_loss" not in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])
        eval_accs.append(log["eval_accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_steps, train_losses, label="Train loss")
axes[0].plot(eval_steps,  eval_losses,  label="Val loss")
axes[0].axhline(y=min(eval_losses), color="gray", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(eval_steps, eval_accs, marker="o", color="green")
axes[1].axhline(y=0.566, color="red", linestyle="--", alpha=0.7, label="Baseline (56.6%)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy vs. baseline")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Save results
os.makedirs(OUTPUT_DIR, exist_ok=True)
summary = {
    "model":    MODEL_NAME,
    "data":     "Refined Dataset (5,567 samples)",
    "accuracy": float(final_acc),
    "improvement_over_baseline_pp": round((final_acc - 0.566) * 100, 2),
    "timestamp": datetime.now().isoformat(),
    "hyperparameters": {
        "lr": LR, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
        "class_weights": [3.25, 1.96, 1.0],
    },
}
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Results saved to {OUTPUT_DIR}/results.json")